# Clean evaluation — every model, no attack

The baseline of the whole suite: all eight models (DNN, AT-rFGSM, AT-MaxMA, KDE, DLA, DNN+, ICNN, PAD) loaded and checked on clean data — predictions on the test set, the detectors' thresholds (τ), and their responses to benign samples, malware samples, and malware with tiny (+0.01 / +0.05) perturbations. Every later number in this section is measured against this clean behaviour.

**Where this fits:** this is the culminating section — the organised version of everything before it. The defences come from `04-defenses/`, the attacks from `03-attacks/` (σ itself from `04-defenses/sigma_zero_attack.ipynb`), and the protocol is uniform: same data, same manipulation mask, same scoring, results saved as tensor archives for the comparison notebooks. The final published method lives in the separate sigma-binary repository.

*Consolidated from `Effectiveness_detectors_no_attack.ipynb` (originally in `thesis`). Outputs cleared.*

## Step 1 — Setup: data, the feature mask, the model code

# Loading data

In [ ]:
import torch
import torch.nn as nn

# Assuming you are using a GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
import gdown

download_links = [
                  'https://drive.google.com/uc?id=1dkoFDyJMuGewWX-LE5rb7zkRU6XkSa2z',
                  'https://drive.google.com/uc?id=1oHJBbqyKybmlFFRZ7-uftn77Vg0Hr_A5',
                  'https://drive.google.com/uc?id=1YwxR5CCNnG5gjeFBlBA8mEfDQM22mLj_',
                  'https://drive.google.com/uc?id=1XB5jdefjnGSxrb27SO-65JZpz5UcsNh2',
                  'https://drive.google.com/uc?id=1Ahw4prIhm322LU53Yhkju5ixd6q1KCJ6',
                  'https://drive.google.com/uc?id=18_YhP6qHHmu52CfANYD2LqxrOzq5ECxl',
                  'https://drive.google.com/uc?id=140wKHyRpKurSuvEOswKdyH3kKRPg8VW_',
                  'https://drive.google.com/uc?id=1pTokv8xKvZoUo8kejJc7N757iNfHmt2s',
                  'https://drive.google.com/uc?id=107sDEWVIJZJWC1SIWASEepdRXB3VWlCZ',
                  'https://drive.google.com/uc?id=1sFBZ3A8VQ-iDjmwASg4u5uHQQR8kVJy9',


]


output_filepath = '/content/'
for link in download_links:
  gdown.download(link, output_filepath)

In [ ]:
import zipfile

# Define the path to the zip file and the extraction path
zip_file_path = '/content/datasets.zip'
extract_to_path = '/content/'

# Unzip the file
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_to_path)

In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset

# Load tensors
train_features = torch.load('train_features.pt', weights_only=True).float()
train_labels = torch.load('train_labels.pt', weights_only=True).long()
val_features = torch.load('val_features.pt', weights_only=True).float()
val_labels = torch.load('val_labels.pt', weights_only=True).long()
test_features = torch.load('test_features.pt', weights_only=True).float()
test_labels = torch.load('test_labels.pt', weights_only=True).long()

# Print data shapes for verification
print(f"Train Features Shape: {train_features.shape}")
print(f"Train Labels Shape: {train_labels.shape}")
print(f"Validation Features Shape: {val_features.shape}")
print(f"Validation Labels Shape: {val_labels.shape}")
print(f"Test Features Shape: {test_features.shape}")
print(f"Test Labels Shape: {test_labels.shape}")

# Create datasets
train_dataset = TensorDataset(train_features, train_labels)
val_dataset = TensorDataset(val_features, val_labels)
test_dataset = TensorDataset(test_features, test_labels)

# Set batch size
batch_size = 1000
# Create DataLoaders
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(dataset=val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

# Optional: Verify DataLoader sizes
print('----------------------------------------------')
print(f"Train Loader Batch Count: {len(train_loader)}")
print(f"Validation Loader Batch Count: {len(val_loader)}")
print(f"Test Loader Batch Count: {len(test_loader)}")

In [ ]:
# Function to count samples for each label
def count_labels(labels):
    count_0 = (labels == 0).sum().item()  # Count samples with label = 0
    count_1 = (labels == 1).sum().item()  # Count samples with label = 1
    return count_0, count_1

# Count labels for each dataset
train_count_0, train_count_1 = count_labels(train_labels)
val_count_0, val_count_1 = count_labels(val_labels)
test_count_0, test_count_1 = count_labels(test_labels)

# Calculate total counts across all datasets
total_count_0 = train_count_0 + val_count_0 + test_count_0
total_count_1 = train_count_1 + val_count_1 + test_count_1

# Print results
print(f"Training set             \t Label 0: {train_count_0}\t Label 1: {train_count_1}")
print(f"Validation set           \t Label 0: {val_count_0}\t Label 1: {val_count_1}")
print(f"Test set                 \t Label 0: {test_count_0}\t Label 1: {test_count_1}")
print(f"Total across all datasets\t Label 0: {total_count_0}\t Label 1: {total_count_1}")

In [ ]:
print(f"The avrage number of feature in Training set: {train_features.sum()/train_features.shape[0]:.2f}")
print(f"The avrage number of feature in Validation set: {val_features.sum()/val_features.shape[0]:.2f}")
print(f"The avrage number of feature in Test set: {test_features.sum()/test_features.shape[0]:.2f}")

In [ ]:
feature_mask = torch.load('manipulation_x.pt', weights_only=True, map_location=device)
feature_mask.sum()

# Models

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

def to_tensor(feature_x=None, labels=None, device='cpu'):

    def _to_torch_tensor(mat):
        mat = torch.DoubleTensor(mat)
        return mat

    feature_x = _to_torch_tensor(feature_x).to(device)
    if labels is None:
        return feature_x
    else:
        labels = torch.LongTensor(labels).to(device)
        return feature_x, labels

def to_device(feature_x=None, labels=None, device='cpu'):
    if feature_x is not None:
        assert isinstance(feature_x, torch.Tensor)
        feature_x = feature_x.to(device)
    if labels is not None:
        assert isinstance(labels, torch.Tensor)
        labels = labels.to(device)
    return feature_x, labels

class MalwareDetectionDNN(nn.Module):
    def __init__(self, input_size, n_classes, device='cpu', name='DNN', dropout = 0.6, alpha = 0.2, smooth = False):
        """
        Construct malware detector

        Parameters
        ----------
        @param input_size: Integer, the dimentionality number of input vector
        @param n_classes: Integer, the number of classes, n=2
        @param device: String, 'cpu' or 'cuda'
        @param name: String, model name
        """
        super(MalwareDetectionDNN, self).__init__()
        self.input_size = input_size
        self.n_classes = n_classes
        self.device = device
        self.name = name
        self.dense_hidden_units = [200, 200]
        self.dropout = dropout
        self.alpha_ = alpha
        self.smooth = smooth


        self.dense_layers = []
        if len(self.dense_hidden_units) >= 1:
            self.dense_layers.append(nn.Linear(self.input_size, self.dense_hidden_units[0]))
        else:
            raise ValueError("Expect at least one hidden layer.")
        for i in range(len(self.dense_hidden_units[0:-1])):
            self.dense_layers.append(nn.Linear(self.dense_hidden_units[i],  # start from idx=1
                                               self.dense_hidden_units[i + 1]))
        self.dense_layers.append(nn.Linear(self.dense_hidden_units[-1], self.n_classes))
        # registration
        for idx_i, dense_layer in enumerate(self.dense_layers):
            self.add_module('nn_model_layer_{}'.format(idx_i), dense_layer)

        if self.smooth:
            self.activation_func = F.selu  # partial(F.elu, alpha=self.alpha_)
        else:
            self.activation_func = F.relu


    def forward(self, x):
        """
        Go through the neural network

        Parameters
        ----------
        @param x: 2D tensor, feature representation
        """
        for dense_layer in self.dense_layers[:-1]:
            x = self.activation_func(dense_layer(x))

        latent_representation = F.dropout(x, self.dropout, training=self.training)
        logits = self.dense_layers[-1](latent_representation)
        return logits

    def inference(self, test_data_producer):
        confidences = []
        gt_labels = []
        self.eval()
        with torch.no_grad():
            for x, y in test_data_producer:
                x, y = to_device(x.double(), y.long(), self.device)
                logits = self.forward(x)
                confidences.append(F.softmax(logits, dim=-1))
                gt_labels.append(y)
        confidences = torch.vstack(confidences)
        gt_labels = torch.cat(gt_labels, dim=0)
        return confidences, gt_labels


    def inference_batch_wise(self, x):
        """
        support malware samples solely
        """
        assert isinstance(x, torch.Tensor)
        logit = self.forward(x)
        return torch.softmax(logit, dim=-1).detach().cpu().numpy(), np.ones((logit.size()[0],))

    def predict(self, test_data_producer, indicator_masking=True):
        """
        predict labels and conduct evaluation

        Parameters
        --------
        @param test_data_producer, torch.DataLoader
        """
        # evaluation
        confidence, y_true = self.inference(test_data_producer)
        y_pred = confidence.argmax(1).cpu().numpy()
        y_true = y_true.cpu().numpy()

        def measurement(_y_true, _y_pred):
            from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, balanced_accuracy_score
            accuracy = accuracy_score(_y_true, _y_pred)
            b_accuracy = balanced_accuracy_score(_y_true, _y_pred)
            print(f"The accuracy on the test dataset is {(accuracy * 100):.2f}%")
            print(f"The balanced accuracy on the test dataset is {(b_accuracy * 100):.2f}%")

            if np.any([np.all(_y_true == i) for i in range(self.n_classes)]):
                return

            tn, fp, fn, tp = confusion_matrix(_y_true, _y_pred).ravel()
            fpr = fp / float(tn + fp)
            fnr = fn / float(tp + fn)
            f1 = f1_score(_y_true, _y_pred, average='binary')
            #print("Other evaluation metrics we may need:")
            print(f"False Negative Rate (FNR) is {fnr * 100:.2f}%, False Positive Rate (FPR) is {fpr * 100:.2f}%, F1 score is {f1 * 100:.2f}%")

        measurement(y_true, y_pred)


    def load(self, model_save_path):
        """
        load model parameters from disk
        """
        self.load_state_dict(torch.load(model_save_path, map_location=torch.device(device)))

    # loading rFGSM
    def load2(self, model_save_path):
        ckpt = torch.load(model_save_path, map_location=torch.device(device))
        self.load_state_dict(ckpt['model'])

In [ ]:
import numpy as np

class DetectorTemplate(object):
    def __init__(self):
        self.tau = None
        self.is_detector_enabled = True

    def forward(self, x):
        """class prediction & density estimation"""
        raise NotImplementedError

    def get_threshold(self):
        """calculate the threshold for rejecting outliers"""
        raise NotImplementedError

    def get_tau_sample_wise(self):
        raise NotImplementedError

    def indicator(self):
        """return a vector of boolean flags indicating whether reject a sample"""
        raise NotImplementedError


def to_tensor(feature_x=None, labels=None, device='cpu'):

    def _to_torch_tensor(mat):
        mat = torch.DoubleTensor(mat)
        return mat

    feature_x = _to_torch_tensor(feature_x).to(device)
    if labels is None:
        return feature_x
    else:
        labels = torch.LongTensor(labels).to(device)
        return feature_x, labels

def to_device(feature_x=None, labels=None, device='cpu'):
    if feature_x is not None:
        assert isinstance(feature_x, torch.Tensor)
        feature_x = feature_x.to(device)
    if labels is not None:
        assert isinstance(labels, torch.Tensor)
        labels = labels.to(device)
    return feature_x, labels


class KernelDensityEstimation(nn.Module, DetectorTemplate):
    """
    kernel density estimation upon the penultimate layer

    parameters
    -------------
    @param model, torch.nn.Module, an instantiation of model object
    @param bandwidth, float, variance of Gaussian density function
    @param n_classes, integer, number of classes
    @param ratio, float [0,1], ratio for computing the threshold
    """

    def __init__(self, model, n_centers=1000, bandwidth=20., n_classes=2, ratio=0.95, name='KDE', device='cpu'):
        super(KernelDensityEstimation, self).__init__()
        assert isinstance(model, torch.nn.Module)
        self.model = model
        self.device = device
        self.name = name
        self.n_centers = n_centers
        self.bandwidth = bandwidth
        self.n_classes = n_classes
        self.ratio = ratio
        self.gaussian_means = None

        self.tau = nn.Parameter(torch.zeros([self.n_classes, ], device=self.device), requires_grad=False)

    def forward(self, x):
        for dense_layer in self.model.dense_layers[:-1]:
            x = self.model.activation_func(dense_layer(x))
        logits = self.model.dense_layers[-1](x)
        x_prob = self.forward_g(x, logits.argmax(1).detach())
        return logits, x_prob

    def forward_f(self, x):
        for dense_layer in self.model.dense_layers[:-1]:
            x = self.model.activation_func(dense_layer(x))
        logits = self.model.dense_layers[-1](x)
        return logits, x

    def forward_g(self, x_hidden, y_pred):
        """
        1 / |X_| * \sum(exp(||x - x_||_2^2/sigma^2))

        parameters
        -----------
        @param x_hidden, torch.tensor, node hidden representation
        @param y_pred, torch.tensor, prediction
        """
        size = x_hidden.size()[0]
        dist = [torch.sum(torch.square(means.unsqueeze(dim=0) - x_hidden.unsqueeze(dim=1)), dim=-1) for means in
                self.gaussian_means]
        kd = torch.stack([torch.mean(torch.exp(-d / self.bandwidth ** 2), dim=-1) for d in dist], dim=1)
        # return p(x|y=y_pred)
        return -1 * kd[torch.arange(size), y_pred]

    def get_threshold(self, validation_data_producer, ratio=None):
        """
        get the threshold for density estimation
        :@param validation_data_producer: Object, an iterator for producing validation dataset
        """
        ratio = ratio if ratio is not None else self.ratio
        assert 0 <= ratio <= 1
        self.eval()
        probabilities = []
        gt_labels = []
        with torch.no_grad():
            for x_val, y_val in validation_data_producer:
                x_val, y_val = to_tensor(x_val.double(), y_val.long(), self.device)
                logits, x_prob = self.forward(x_val)
                probabilities.append(x_prob)
                gt_labels.append(y_val)
            prob = torch.cat(probabilities, dim=0)
            gt_labels = torch.cat(gt_labels)
            for i in range(self.n_classes):
                prob_x_y = prob[gt_labels == i]
                s, _ = torch.sort(prob_x_y)
                self.tau[i] = s[int((s.shape[0] - 1) * ratio)]

    def predict(self, test_data_producer, indicator_masking=True):
        # evaluation on detector & indicator
        y_cent, x_prob, y_true = self.inference(test_data_producer)
        y_pred = y_cent.argmax(1).cpu().numpy()
        y_true = y_true.cpu().numpy()
        indicator_flag = self.indicator(x_prob, y_pred).cpu().numpy()

        def measurement(_y_true, _y_pred):
            from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, balanced_accuracy_score
            accuracy = accuracy_score(_y_true, _y_pred)
            b_accuracy = balanced_accuracy_score(_y_true, _y_pred)
            print(f"The accuracy on the test dataset is {(accuracy * 100):.2f}%")
            print(f"The balanced accuracy on the test dataset is {(b_accuracy * 100):.2f}%")

            if np.any([np.all(_y_true == i) for i in range(self.n_classes)]):
                return

            tn, fp, fn, tp = confusion_matrix(_y_true, _y_pred).ravel()
            fpr = fp / float(tn + fp)
            fnr = fn / float(tp + fn)
            f1 = f1_score(_y_true, _y_pred, average='binary')
            #print("Other evaluation metrics we may need:")
            print(f"False Negative Rate (FNR) is {fnr * 100:.2f}%, False Positive Rate (FPR) is {fpr * 100:.2f}%, F1 score is {f1 * 100:.2f}%")



        if indicator_masking:
            flag_of_retaining = indicator_flag  # excluding the examples with ``not sure'' response
            y_pred = y_pred[flag_of_retaining]
            y_true = y_true[flag_of_retaining]
            ExS = (len(y_cent) - len(y_true))/len(y_cent)
            print('Deferred option : ')
            print('The threshold is {}'.format(self.tau))
            print(f"Percentage of excluded samples is {ExS * 100:.2f}%")
            measurement(y_true, y_pred)

        else:
            # instead filtering out examples, here resets the prediction as 1 (test model on perturbed examples)
            y_pred[~indicator_flag] = 1.
            print('Conservative option : ')
            print('The threshold is {}'.format(self.tau))
            measurement(y_true, y_pred)





    def eval(self):
        self.model.eval()

    def inference(self, test_data_producer):
        y_cent, x_prob = [], []
        gt_labels = []
        self.eval()
        with torch.no_grad():
            for x, y in test_data_producer:
                x, y = x.to(device), y.to(device)
                logits_f, logits_g = self.forward(x)
                y_cent.append(F.softmax(logits_f, dim=-1))
                x_prob.append(logits_g)
                gt_labels.append(y)

        gt_labels = torch.cat(gt_labels, dim=0)
        y_cent = torch.cat(y_cent, dim=0)
        x_prob = torch.cat(x_prob, dim=0)
        return y_cent, x_prob, gt_labels

    def inference_batch_wise(self, x):
        assert isinstance(x, torch.Tensor)

        logits, x_prob = self.forward(x)
        return torch.softmax(logits, dim=-1).detach().cpu().numpy(), x_prob.detach().cpu().numpy()

    def get_tau_sample_wise(self, y_pred=None):
        return self.tau[y_pred]

    def indicator(self, x_prob, y_pred=None):
        assert y_pred is not None
        if isinstance(x_prob, np.ndarray):
            x_prob = torch.tensor(x_prob, device=self.device)
            return (x_prob <= self.get_tau_sample_wise(y_pred)).cpu().numpy()
        elif isinstance(x_prob, torch.Tensor):
            return x_prob <= self.get_tau_sample_wise(y_pred)
        else:
            raise TypeError("Tensor or numpy.ndarray are expected.")
        # res = probability.reshape(-1, 1).repeat_interleave(2, dim=1) >= self.tau
        # return res[torch.arange(res.size()[0]), y_pred]


    def load(self, model_save_path):
        ckpt = torch.load(model_save_path, map_location=torch.device(device))
        self.gaussian_means = ckpt['gaussian_means']
        self.tau = ckpt['tau']
        self.model.load_state_dict(ckpt['base_model'])

In [ ]:
import time
import warnings
import os.path as path
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F


import numpy as np


class DetectorTemplate(object):
    def __init__(self):
        self.tau = None
        self.is_detector_enabled = True

    def forward(self, x):
        """class prediction & density estimation"""
        raise NotImplementedError

    def get_threshold(self):
        """calculate the threshold for rejecting outliers"""
        raise NotImplementedError

    def get_tau_sample_wise(self):
        raise NotImplementedError

    def indicator(self):
        """return a vector of boolean flags indicating whether reject a sample"""
        raise NotImplementedError



def to_tensor(feature_x=None, labels=None, device='cpu'):

    def _to_torch_tensor(mat):
        mat = torch.DoubleTensor(mat)
        return mat

    feature_x = _to_torch_tensor(feature_x).to(device)
    if labels is None:
        return feature_x
    else:
        labels = torch.LongTensor(labels).to(device)
        return feature_x, labels

def to_device(feature_x=None, labels=None, device='cpu'):
    if feature_x is not None:
        assert isinstance(feature_x, torch.Tensor)
        feature_x = feature_x.to(device)
    if labels is not None:
        assert isinstance(labels, torch.Tensor)
        labels = labels.to(device)
    return feature_x, labels



class AMalwareDetectionDLA(nn.Module, DetectorTemplate):
    def __init__(self, md_nn_model, input_size, n_classes, ratio=0.95,
                 device='cpu', name='', dropout = 0.6, alpha = 0.2, smooth = False):
        nn.Module.__init__(self)
        DetectorTemplate.__init__(self)
        self.input_size = input_size
        self.n_classes = n_classes
        self.ratio = ratio
        self.device = device
        self.name = name
        self.dense_hidden_units = [200, 200]
        self.dropout = dropout
        self.alpha_ = alpha
        self.smooth = smooth


        # malware detector
        if md_nn_model is not None and isinstance(md_nn_model, nn.Module):
            self.md_nn_model = md_nn_model
            self.is_fitting_md_model = False  # the model is trained by default
        else:
            self.md_nn_model = MalwareDetectionDNN(self.input_size,
                                                   n_classes,
                                                   self.device,
                                                   name)
            self.is_fitting_md_model = True

        assert len(self.dense_hidden_units) >= 1, "Expected at least one hidden layer."
        self.alarm_nn_model = TorchAlarm(input_size=sum(self.md_nn_model.dense_hidden_units))

        self.tau = nn.Parameter(torch.zeros([1, ], device=self.device), requires_grad=False)

    def forward(self, x):
        hidden_activations = []
        for dense_layer in self.md_nn_model.dense_layers[:-1]:
            x = self.md_nn_model.activation_func(dense_layer(x))
            hidden_activations.append(x)
        logits = self.md_nn_model.dense_layers[-1](x)
        hidden_activations = torch.cat(hidden_activations, dim=-1)
        x_prob = self.alarm_nn_model(hidden_activations).reshape(-1)
        return logits, x_prob

    def predict(self, test_data_producer, indicator_masking=True):
        """
        predict labels and conduct evaluation on detector & indicator

        Parameters
        --------
        @param test_data_producer, torch.DataLoader
        @param indicator_masking, whether filtering out the examples with low density or masking their values
        """
        y_cent, x_prob, y_true = self.inference(test_data_producer)
        y_pred = y_cent.argmax(1).cpu().numpy()
        y_true = y_true.cpu().numpy()
        indicator_flag = self.indicator(x_prob).cpu().numpy()

        def measurement(_y_true, _y_pred):
            from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, balanced_accuracy_score
            accuracy = accuracy_score(_y_true, _y_pred)
            b_accuracy = balanced_accuracy_score(_y_true, _y_pred)
            print(f"The accuracy on the test dataset is {(accuracy * 100):.2f}%")
            print(f"The balanced accuracy on the test dataset is {(b_accuracy * 100):.2f}%")

            if np.any([np.all(_y_true == i) for i in range(self.n_classes)]):
                return

            tn, fp, fn, tp = confusion_matrix(_y_true, _y_pred).ravel()
            fpr = fp / float(tn + fp)
            fnr = fn / float(tp + fn)
            f1 = f1_score(_y_true, _y_pred, average='binary')
            #print("Other evaluation metrics we may need:")
            print(f"False Negative Rate (FNR) is {fnr * 100:.2f}%, False Positive Rate (FPR) is {fpr * 100:.2f}%, F1 score is {f1 * 100:.2f}%")



        if indicator_masking:
            flag_of_retaining = indicator_flag  # excluding the examples with ``not sure'' response
            y_pred = y_pred[flag_of_retaining]
            y_true = y_true[flag_of_retaining]
            ExS = (len(y_cent) - len(y_true))/len(y_cent)
            print('Deferred option : ')
            print('The threshold is {:.5}'.format(self.tau.item()))
            print(f"Percentage of excluded samples is {ExS * 100:.2f}%")
            measurement(y_true, y_pred)

        else:
            # instead filtering out examples, here resets the prediction as 1 (test model on perturbed examples)
            y_pred[~indicator_flag] = 1.
            print('Conservative option : ')
            print('The threshold is {:.5}'.format(self.tau.item()))
            measurement(y_true, y_pred)


    def inference(self, test_data_producer):
        y_cent, x_prob = [], []
        gt_labels = []
        self.eval()
        with torch.no_grad():
            for x, y in test_data_producer:
                x, y = to_device(x.double(), y.long(), self.device)
                logits, x_logits = self.forward(x)
                y_cent.append(F.softmax(logits, dim=-1))
                x_prob.append(x_logits)
                gt_labels.append(y)

        gt_labels = torch.cat(gt_labels, dim=0)
        y_cent = torch.cat(y_cent, dim=0)
        x_prob = torch.cat(x_prob, dim=0)
        return y_cent, x_prob, gt_labels

    def inference_batch_wise(self, x):
        assert isinstance(x, torch.Tensor)
        self.eval()
        logits, x_prob = self.forward(x)
        return torch.softmax(logits, dim=-1).detach().cpu().numpy(), x_prob.detach().cpu().numpy()

    def get_tau_sample_wise(self, y_pred=None):
        return self.tau

    def indicator(self, x_prob, y_pred=None):
        """
        Return 'True' if a sample is original, and otherwise 'False' is returned.
        """
        if isinstance(x_prob, np.ndarray):
            x_prob = torch.tensor(x_prob, device=self.device)
            return (x_prob <= self.tau).cpu().numpy()
        elif isinstance(x_prob, torch.Tensor):
            return x_prob <= self.tau
        else:
            raise TypeError("Tensor or numpy.ndarray are expected.")

    def get_threshold(self, validation_data_producer, ratio=None):
        """
        get the threshold for adversary detection
        :@param validation_data_producer: Object, an iterator for producing validation dataset
        """
        self.eval()
        ratio = ratio if ratio is not None else self.ratio
        assert 0 <= ratio <= 1
        probabilities = []
        with torch.no_grad():
            for x_val, y_val in validation_data_producer:
                x_val, y_val = to_tensor(x_val.double(), y_val.long(), self.device)
                _1, x_prob = self.forward(x_val)
                probabilities.append(x_prob)
            s, _ = torch.sort(torch.cat(probabilities, dim=0))
            i = int((s.shape[0] - 1) * ratio)
            assert i >= 0
            self.tau[0] = s[i]

    def load(self, model_save_path):
        self.load_state_dict(torch.load(model_save_path, map_location=torch.device(device)))


class TorchAlarm(torch.nn.Module):
    def __init__(self, input_size):
        super().__init__()

        self.layers = torch.nn.ModuleList([
            torch.nn.Linear(input_size, 112),
            torch.nn.ReLU(),
            torch.nn.Linear(112, 100),
            torch.nn.ReLU(),
            torch.nn.Linear(100, 300),
            torch.nn.ReLU(),
            torch.nn.Linear(300, 200),
            torch.nn.ReLU(),
            torch.nn.Linear(200, 77),
            torch.nn.ReLU(),
            torch.nn.Linear(77, 1),
        ])

    def __call__(self, x, training=False):
        if not isinstance(x, torch.Tensor):
            x = torch.tensor(x, dtype=torch.float32)
        for layer in self.layers:
            x = layer(x)
        return x

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np

class DetectorTemplate(object):
    def __init__(self):
        self.tau = None
        self.is_detector_enabled = True

    def forward(self, x):
        """class prediction & density estimation"""
        raise NotImplementedError

    def get_threshold(self):
        """calculate the threshold for rejecting outliers"""
        raise NotImplementedError

    def get_tau_sample_wise(self):
        raise NotImplementedError

    def indicator(self):
        """return a vector of boolean flags indicating whether reject a sample"""
        raise NotImplementedError



def to_tensor(feature_x=None, labels=None, device='cpu'):

    def _to_torch_tensor(mat):
        mat = torch.DoubleTensor(mat)
        return mat

    feature_x = _to_torch_tensor(feature_x).to(device)
    if labels is None:
        return feature_x
    else:
        labels = torch.LongTensor(labels).to(device)
        return feature_x, labels

def to_device(feature_x=None, labels=None, device='cpu'):
    if feature_x is not None:
        assert isinstance(feature_x, torch.Tensor)
        feature_x = feature_x.to(device)
    if labels is not None:
        assert isinstance(labels, torch.Tensor)
        labels = labels.to(device)
    return feature_x, labels







class AMalwareDetectionDNNPlus(nn.Module, DetectorTemplate):
    def __init__(self, md_nn_model, input_size, n_classes, ratio=0.95,
                 device='cpu', name='', dropout = 0.6, alpha = 0.2, smooth = False):
        nn.Module.__init__(self)
        DetectorTemplate.__init__(self)
        self.input_size = input_size
        self.n_classes = n_classes
        self.ratio = ratio
        self.device = device
        self.name = name
        self.dense_hidden_units = [200, 200]
        self.dropout = dropout
        self.alpha_ = alpha
        self.smooth = smooth

        # malware detector
        if md_nn_model is not None and isinstance(md_nn_model, nn.Module):
            self.md_nn_model = md_nn_model
            self.is_fitting_md_model = False  # the model is trained by default
        else:
            self.md_nn_model = MalwareDetectionDNN(self.input_size,
                                                   self.n_classes,
                                                   self.device,
                                                   name)
            self.is_fitting_md_model = True

        self.amd_nn_plus = MalwareDetectionDNN(self.input_size,
                                               self.n_classes + 1,
                                               self.device,
                                               name)

        self.tau = nn.Parameter(torch.zeros([1, ], device=self.device), requires_grad=False)


    def forward(self, x):
        logits = self.amd_nn_plus(x)
        logits -= torch.amax(logits, dim=-1, keepdim=True).detach()  # increasing the stability, which might be helpful
        return logits, torch.softmax(logits, dim=-1)[:, -1]

    def predict(self, test_data_producer, indicator_masking=True):
        """
        predict labels and conduct evaluation on detector & indicator

        Parameters
        --------
        @param test_data_producer, torch.DataLoader
        @param indicator_masking, whether filtering out the examples with low density or masking their values
        """
        y_cent, x_prob, y_true = self.inference(test_data_producer)
        y_pred = y_cent.argmax(1).cpu().numpy()
        y_true = y_true.cpu().numpy()
        indicator_flag = self.indicator(x_prob).cpu().numpy()

        def measurement(_y_true, _y_pred):
            from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, balanced_accuracy_score
            accuracy = accuracy_score(_y_true, _y_pred)
            b_accuracy = balanced_accuracy_score(_y_true, _y_pred)
            print(f"The accuracy on the test dataset is {(accuracy * 100):.2f}%")
            print(f"The balanced accuracy on the test dataset is {(b_accuracy * 100):.2f}%")

            if np.any([np.all(_y_true == i) for i in range(self.n_classes)]):
                return

            tn, fp, fn, tp = confusion_matrix(_y_true, _y_pred).ravel()
            fpr = fp / float(tn + fp)
            fnr = fn / float(tp + fn)
            f1 = f1_score(_y_true, _y_pred, average='binary')
            #print("Other evaluation metrics we may need:")
            print(f"False Negative Rate (FNR) is {fnr * 100:.2f}%, False Positive Rate (FPR) is {fpr * 100:.2f}%, F1 score is {f1 * 100:.2f}%")



        if indicator_masking:
            flag_of_retaining = indicator_flag  # excluding the examples with ``not sure'' response
            y_pred = y_pred[flag_of_retaining]
            y_true = y_true[flag_of_retaining]
            ExS = (len(y_cent) - len(y_true))/len(y_cent)
            print('Deferred option : ')
            print('The threshold is {:.5}'.format(self.tau.item()))
            print(f"Percentage of excluded samples is {ExS * 100:.2f}%")
            measurement(y_true, y_pred)

        else:
            # instead filtering out examples, here resets the prediction as 1 (test model on perturbed examples)
            y_pred[~indicator_flag] = 1.
            print('Conservative option : ')
            print('The threshold is {:.5}'.format(self.tau.item()))
            measurement(y_true, y_pred)



    def inference(self, test_data_producer):
        y_cent, x_prob = [], []
        gt_labels = []
        self.eval()
        with torch.no_grad():
            for x, y in test_data_producer:
                x, y = to_device(x.double(), y.long(), self.device)
                logits, x_cent = self.forward(x)
                y_cent.append(F.softmax(logits, dim=-1)[:, :2])
                x_prob.append(x_cent)
                gt_labels.append(y)

        gt_labels = torch.cat(gt_labels, dim=0)
        y_cent = torch.cat(y_cent, dim=0)
        x_prob = torch.cat(x_prob, dim=0)
        return y_cent, x_prob, gt_labels

    def inference_batch_wise(self, x):
        assert isinstance(x, torch.Tensor)
        self.eval()
        logits, g = self.forward(x)
        x_cent = torch.softmax(logits, dim=-1).detach().cpu().numpy()[:, :2]

        return x_cent, g.detach().cpu().numpy()

    def get_tau_sample_wise(self, y_pred=None):
        return self.tau

    def indicator(self, x_prob, y_pred=None):
        """
        Return 'True' if a sample is original, and otherwise 'False' is returned.
        """
        if isinstance(x_prob, np.ndarray):
            x_prob = torch.tensor(x_prob, device=self.device)
            return (x_prob <= self.tau).cpu().numpy()
        elif isinstance(x_prob, torch.Tensor):
            return x_prob <= self.tau
        else:
            raise TypeError("Tensor or numpy.ndarray are expected.")

    def get_threshold(self, validation_data_producer, ratio=None):
        """
        get the threshold for adversary detection
        :@param validation_data_producer: Object, an iterator for producing validation dataset
        """
        self.eval()
        ratio = ratio if ratio is not None else self.ratio
        assert 0 <= ratio <= 1
        probabilities = []
        with torch.no_grad():
            for x_val, y_val in validation_data_producer:
                x_val, y_val = to_tensor(x_val.double(), y_val.long(), self.device)
                _1, x_cent = self.forward(x_val)
                probabilities.append(x_cent)
            s, _ = torch.sort(torch.cat(probabilities, dim=0))
            i = int((s.shape[0] - 1) * ratio)
            assert i >= 0
            self.tau[0] = s[i]

    def load(self, model_save_path):
        self.load_state_dict(torch.load(model_save_path, map_location=torch.device(device)))

In [ ]:
import time
import warnings
import os.path as path
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np


class DetectorTemplate(object):
    def __init__(self):
        self.tau = None
        self.is_detector_enabled = True

    def forward(self, x):
        """class prediction & density estimation"""
        raise NotImplementedError

    def get_threshold(self):
        """calculate the threshold for rejecting outliers"""
        raise NotImplementedError

    def get_tau_sample_wise(self):
        raise NotImplementedError

    def indicator(self):
        """return a vector of boolean flags indicating whether reject a sample"""
        raise NotImplementedError


def to_tensor(feature_x=None, labels=None, device='cpu'):

    def _to_torch_tensor(mat):
        mat = torch.DoubleTensor(mat)
        return mat

    feature_x = _to_torch_tensor(feature_x).to(device)
    if labels is None:
        return feature_x
    else:
        labels = torch.LongTensor(labels).to(device)
        return feature_x, labels

def to_device(feature_x=None, labels=None, device='cpu'):
    if feature_x is not None:
        assert isinstance(feature_x, torch.Tensor)
        feature_x = feature_x.to(device)
    if labels is not None:
        assert isinstance(labels, torch.Tensor)
        labels = labels.to(device)
    return feature_x, labels


class AdvMalwareDetectorICNN(nn.Module, DetectorTemplate):
    def __init__(self, md_nn_model, input_size, n_classes, ratio=0.95,
                 device='cpu', name='ICNN', ):
        nn.Module.__init__(self)
        DetectorTemplate.__init__(self)
        self.md_nn_model = md_nn_model
        self.input_size = input_size
        self.n_classes = n_classes
        self.ratio = ratio
        self.device = device
        self.name = name
        self.dense_hidden_units = [200, 200]


        if hasattr(self.md_nn_model, 'smooth'):
            if not self.md_nn_model.smooth:  # non-smooth, exchange it
                for name, child in self.md_nn_model.named_children():
                    if isinstance(child, nn.ReLU):
                        self.md_nn_model._modules['relu'] = nn.SELU()
        else:
            for name, child in self.md_nn_model.named_children():
                if isinstance(child, nn.ReLU):
                    self.md_nn_model._modules['relu'] = nn.SELU()
        self.md_nn_model = self.md_nn_model.to(self.device)

        # input convex neural network
        self.non_neg_dense_layers = []
        if len(self.dense_hidden_units) < 1:
            raise ValueError("Expect at least one hidden layer.")
        for i in range(len(self.dense_hidden_units[0:-1])):
            self.non_neg_dense_layers.append(nn.Linear(self.dense_hidden_units[i],  # start from idx=1
                                                       self.dense_hidden_units[i + 1],
                                                       bias=False))
        self.non_neg_dense_layers.append(nn.Linear(self.dense_hidden_units[-1], 1, bias=False))
        # registration
        for idx_i, dense_layer in enumerate(self.non_neg_dense_layers):
            self.add_module('non_neg_layer_{}'.format(idx_i), dense_layer)

        self.dense_layers = []
        self.dense_layers.append(nn.Linear(self.input_size, self.dense_hidden_units[0]))
        for i in range(len(self.dense_hidden_units[1:])):
            self.dense_layers.append(nn.Linear(self.input_size, self.dense_hidden_units[i]))
        self.dense_layers.append(nn.Linear(self.input_size, 1))
        # registration
        for idx_i, dense_layer in enumerate(self.dense_layers):
            self.add_module('layer_{}'.format(idx_i), dense_layer)

        self.tau = nn.Parameter(torch.zeros([1, ], device=self.device), requires_grad=False)


    def forward_f(self, x):
        return self.md_nn_model(x)

    def forward_g(self, x):
        prev_x = None
        for i, dense_layer in enumerate(self.dense_layers):
            x_add = []
            x1 = dense_layer(x)
            x_add.append(x1)
            if prev_x is not None:
                x2 = self.non_neg_dense_layers[i - 1](prev_x)
                x_add.append(x2)
            prev_x = torch.sum(torch.stack(x_add, dim=0), dim=0)
            if i < len(self.dense_layers):
                prev_x = F.selu(prev_x)
        return prev_x.reshape(-1)

    def forward(self, x):
        return self.forward_f(x), self.forward_g(x)

    def predict(self, test_data_producer, indicator_masking=True):
        """
        predict labels and conduct evaluation on detector & indicator

        Parameters
        --------
        @param test_data_producer, torch.DataLoader
        @param indicator_masking, whether filtering out the examples with low density or masking their values
        """
        y_cent, x_prob, y_true = self.inference(test_data_producer)
        y_pred = y_cent.argmax(1).cpu().numpy()
        y_true = y_true.cpu().numpy()
        indicator_flag = self.indicator(x_prob).cpu().numpy()

        def measurement(_y_true, _y_pred):
            from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, balanced_accuracy_score
            accuracy = accuracy_score(_y_true, _y_pred)
            b_accuracy = balanced_accuracy_score(_y_true, _y_pred)
            print(f"The accuracy on the test dataset is {(accuracy * 100):.2f}%")
            print(f"The balanced accuracy on the test dataset is {(b_accuracy * 100):.2f}%")

            if np.any([np.all(_y_true == i) for i in range(self.n_classes)]):
                return

            tn, fp, fn, tp = confusion_matrix(_y_true, _y_pred).ravel()
            fpr = fp / float(tn + fp)
            fnr = fn / float(tp + fn)
            f1 = f1_score(_y_true, _y_pred, average='binary')
            #print("Other evaluation metrics we may need:")
            print(f"False Negative Rate (FNR) is {fnr * 100:.2f}%, False Positive Rate (FPR) is {fpr * 100:.2f}%, F1 score is {f1 * 100:.2f}%")



        if indicator_masking:
            flag_of_retaining = indicator_flag  # excluding the examples with ``not sure'' response
            y_pred = y_pred[flag_of_retaining]
            y_true = y_true[flag_of_retaining]
            ExS = (len(y_cent) - len(y_true))/len(y_cent)
            print('Deferred option : ')
            print('The threshold is {:.5}'.format(self.tau.item()))
            print(f"Percentage of excluded samples is {ExS * 100:.2f}%")
            measurement(y_true, y_pred)

        else:
            # instead filtering out examples, here resets the prediction as 1 (test model on perturbed examples)
            y_pred[~indicator_flag] = 1.
            print('Conservative option : ')
            print('The threshold is {:.5}'.format(self.tau.item()))
            measurement(y_true, y_pred)



    def inference(self, test_data_producer):
        y_cent, x_prob = [], []
        gt_labels = []
        self.eval()
        with torch.no_grad():
            for x, y in test_data_producer:
                x, y = to_device(x.double(), y.long(), self.device)
                logits_f, logits_g = self.forward(x)
                y_cent.append(torch.softmax(logits_f, dim=-1))
                x_prob.append(logits_g)
                gt_labels.append(y)

        gt_labels = torch.cat(gt_labels, dim=0)
        y_cent = torch.cat(y_cent, dim=0)
        x_prob = torch.cat(x_prob, dim=0)
        return y_cent, x_prob, gt_labels


    def inference_batch_wise(self, x):
        assert isinstance(x, torch.Tensor)
        self.eval()
        logits_f, logits_g = self.forward(x)
        return torch.softmax(logits_f, dim=-1).detach().cpu().numpy(), logits_g.detach().cpu().numpy()

    def get_tau_sample_wise(self, y_pred=None):
        return self.tau

    def indicator(self, x_prob, y_pred=None):
        """
        Return 'True' if a sample is original, and otherwise 'False' is returned.
        """
        if isinstance(x_prob, np.ndarray):
            x_prob = torch.tensor(x_prob, device=self.device)
            return (x_prob <= self.tau).cpu().numpy()
        elif isinstance(x_prob, torch.Tensor):
            return x_prob <= self.tau
        else:
            raise TypeError("Tensor or numpy.ndarray are expected.")

    def get_threshold(self, validation_data_producer, ratio=None):
        """
        get the threshold for adversary detection
        :@param validation_data_producer: Object, an iterator for producing validation dataset
        """
        self.eval()
        ratio = ratio if ratio is not None else self.ratio
        assert 0 <= ratio <= 1
        probabilities = []
        with torch.no_grad():
            for x_val, y_val in validation_data_producer:
                x_val, y_val = to_tensor(x_val.double(), y_val.long(), self.device)
                x_logits = self.forward_g(x_val)
                probabilities.append(x_logits)
            s, _ = torch.sort(torch.cat(probabilities, dim=0))
            i = int((s.shape[0] - 1) * ratio)
            assert i >= 0
            self.tau[0] = s[i]

    def reset_threshold(self):
        self.tau[0] = 0.


    def load(self, model_save_path):
        # load model
        self.load_state_dict(torch.load(model_save_path, map_location=torch.device(device)))

    # loading PAD
    def load2(self, model_save_path):
        ckpt = torch.load(model_save_path, map_location=torch.device(device))
        self.load_state_dict(ckpt['model'])

## Step 3 — DNN

# Evaluations

### Load DNN

In [ ]:
model_dnn = MalwareDetectionDNN(10000,2,device=device, name='DNN')
model_dnn.load('/content/model_dnn.pth')

In [ ]:
model_dnn.double().to(device)
model_dnn.predict(test_loader)

In [ ]:
model_dnn.float()

In [ ]:
for x_test, y_test in train_loader:
      x_test, y_test = x_test.to(device), y_test.to(device)
      mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
      ben_x_batch, ben_y_batch = x_test[y_test.squeeze() == 0], y_test[y_test.squeeze() == 0]
      print(ben_x_batch.shape)
      print(mal_x_batch.shape)
      break

In [ ]:
model_dnn(ben_x_batch[:5])

In [ ]:
model_dnn(mal_x_batch[:5])

In [ ]:
model_dnn((mal_x_batch[:5]+0.01))

In [ ]:
model_dnn((mal_x_batch[:5]+0.05))

## Step 4 — AT-rFGSM

## Load rFGSM

In [ ]:
model_rFGSM = MalwareDetectionDNN(10000,2,device=device, name='rFGSM')
model_rFGSM.load2('/content/model_at_pgd.pth')

In [ ]:
model_rFGSM.double().to(device)
model_rFGSM.predict(test_loader)

In [ ]:
model_rFGSM.float()

In [ ]:
for x_test, y_test in test_loader:
      x_test, y_test = x_test.to(device), y_test.to(device)
      mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
      ben_x_batch, ben_y_batch = x_test[y_test.squeeze() == 0], y_test[y_test.squeeze() == 0]
      print(ben_x_batch.shape)
      print(mal_x_batch.shape)
      break

In [ ]:
model_rFGSM(ben_x_batch[:5])

In [ ]:
model_rFGSM(mal_x_batch[:5])

In [ ]:
model_rFGSM((mal_x_batch[:5]+0.01))

In [ ]:
model_rFGSM((mal_x_batch[:5]+0.05))

## Step 5 — AT-MaxMA

## Load MaxMA

In [ ]:
model_MaxMA = MalwareDetectionDNN(10000,2,device=device, name='MaxMA')
model_MaxMA.load2('/content/model_at_ma.pth')

In [ ]:
model_MaxMA.double().to(device)
model_MaxMA.predict(test_loader)

In [ ]:
model_MaxMA.float()

In [ ]:
model_MaxMA(ben_x_batch[:5])

In [ ]:
model_MaxMA(mal_x_batch[:5])

In [ ]:
model_MaxMA((mal_x_batch[:5]+0.01))

In [ ]:
model_MaxMA((mal_x_batch[:5]+0.05))

## Step 6 — KDE

## Loading KDE

In [ ]:
model_dnn = MalwareDetectionDNN(10000,2,device=device, name='DNN')
model_dnn.load('/content/model_dnn.pth')

model_kde = KernelDensityEstimation(model_dnn,n_centers=1000,bandwidth=16.,n_classes=2,ratio=0.95,device=device,name='KDE')
model_kde.load('/content/amd_kde.pth')
model_kde.to(device).float()

In [ ]:
model_kde.predict(test_loader, indicator_masking=True)

In [ ]:
model_kde.predict(test_loader, indicator_masking=False)

In [ ]:
model_kde.tau

In [ ]:
logits, x_prob = model_kde.forward(mal_x_batch[:10])
print(logits, x_prob)

In [ ]:
(x_prob <model_kde.tau[1]).sum()

In [ ]:
model_kde((mal_x_batch[:5]+0.01))

In [ ]:
model_kde((mal_x_batch[:5]+0.05))

## Step 7 — DLA

## Loading DLA

In [ ]:
model_dnn = MalwareDetectionDNN(10000,2,device=device, name='DNN')
model_dnn.load('/content/model_dnn.pth')

model_dla = AMalwareDetectionDLA(model_dnn,input_size=10000,n_classes=2,device=device,name='DLA')
model_dla.load('/content/amd_dla.pth')
model_dla.to(device).float()

In [ ]:
model_dla.double()
model_dla.predict(test_loader, indicator_masking=True)

In [ ]:
model_dla.float()

In [ ]:
model_dla.tau

In [ ]:
model_dla.tau[0].item()

In [ ]:
model_dla(ben_x_batch[:5])

In [ ]:
model_dla(mal_x_batch[:5])

In [ ]:
model_dla((mal_x_batch[:5]+0.01))

In [ ]:
model_dla((mal_x_batch[:5]+0.05))

## Step 8 — DNN+

## Loading DNN_Plus

In [ ]:
model_dnn = MalwareDetectionDNN(10000,2,device=device, name='DNN')
model_dnn.load('/content/model_dnn.pth')

model_dnn_plus = AMalwareDetectionDNNPlus(model_dnn,input_size=10000,n_classes=2,device=device,name='DNNPlus')
model_dnn_plus.load('/content/amd_dnn_plus.pth')
model_dnn_plus.to(device).float()

In [ ]:
model_dnn_plus.double()
model_dnn_plus.predict(test_loader, indicator_masking=True)

In [ ]:
model_dnn_plus.float()

In [ ]:
model_dnn_plus.tau

In [ ]:
model_dnn_plus.tau[0].item()

In [ ]:
model_dnn_plus(ben_x_batch[:5])

In [ ]:
model_dnn_plus(mal_x_batch[:5])

In [ ]:
model_dnn_plus((mal_x_batch[:5]+0.01))

In [ ]:
model_dnn_plus((mal_x_batch[:5]+0.05))

## Step 9 — ICNN

## Loadin ICNN

In [ ]:
model_dnn = MalwareDetectionDNN(10000,2,device=device, name='DNN')
model_dnn.load('/content/model_dnn.pth')

model_icnn = AdvMalwareDetectorICNN(model_dnn,input_size=10000,n_classes=2,device=device,name='ICNN')
model_icnn.load('/content/amd_icnn.pth')
model_icnn.to(device).float()

In [ ]:
model_icnn.double().to(device)
model_icnn.predict(test_loader, indicator_masking=True)

In [ ]:
model_icnn.to(device).float()

In [ ]:
model_icnn.tau

In [ ]:
model_icnn.tau[0].item()

In [ ]:
model_icnn(ben_x_batch[:5])

In [ ]:
model_icnn(mal_x_batch[:5])

In [ ]:
model_icnn((mal_x_batch[:5]+0.01))

In [ ]:
model_icnn((mal_x_batch[:5]+0.05))

## Step 10 — PAD

## Loadin PAD

In [ ]:
model_dnn = MalwareDetectionDNN(10000,2,device=device, name='DNN')
model_dnn.load('/content/model_dnn.pth')

model_pad = AdvMalwareDetectorICNN(model_dnn,input_size=10000,n_classes=2,device=device,name='PAD')
model_pad.load2('/content/amd_pad_ma.pth')
model_pad.to(device).float()

In [ ]:
model_pad.double()
model_pad.predict(test_loader, indicator_masking=True)

In [ ]:
model_pad.to(device).float()

In [ ]:
model_pad.tau

In [ ]:
model_pad.tau[0].item()

In [ ]:
model_pad(ben_x_batch[:5])

In [ ]:
model_pad(mal_x_batch[:5])

In [ ]:
model_pad((mal_x_batch[:5]+0.01))

In [ ]:
model_pad((mal_x_batch[:5]+0.05))